# FIFA World Cup 2026 — Full Tournament Simulation

Visualises all 12 group standings and the complete knockout bracket, with
win-probability annotations derived from 50,000 Monte Carlo simulations.

**Data flow**
```
predict_tournament.py  →  wc2026_upcoming.json  (per-match probabilities)
simulate_tournament.py →  wc2026_simulation.json (bracket + team probs)
```

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

REPO = Path("..")
sys.path.insert(0, str(REPO))

PREDICTIONS = REPO / "data" / "predictions" / "wc2026_upcoming.json"
SIMULATION  = REPO / "data" / "predictions" / "wc2026_simulation.json"

# Run simulation if results file doesn't exist yet
if not SIMULATION.exists():
    import subprocess, sys
    subprocess.run([sys.executable, str(REPO / "scripts" / "simulate_tournament.py"), "--n-sims", "50000"], check=True)

with open(PREDICTIONS) as f:
    preds: list[dict] = json.load(f)
with open(SIMULATION) as f:
    sim_data = json.load(f)

probs: dict[str, dict] = sim_data["probabilities"]
bracket: dict = sim_data["bracket"]

print(f"Loaded {len(preds)} matches, {len(probs)} teams")

Loaded 104 matches, 48 teams


## Group Stage Standings

Expected standings based on the most-likely outcome for each match.

In [2]:
group_standings: dict = bracket["group_standings"]
group_ids = sorted(group_standings.keys())  # GROUP_A … GROUP_L

N_COLS = 4
N_ROWS = 3

fig = make_subplots(
    rows=N_ROWS, cols=N_COLS,
    subplot_titles=[gid.replace("GROUP_", "Group ") for gid in group_ids],
    vertical_spacing=0.08,
    horizontal_spacing=0.04,
)

QUAL_COLORS = ["#1a9641", "#a6d96a", "#fdae61", "#d73027"]
QUAL_LABELS = ["1st — Advances", "2nd — Advances", "3rd — Maybe", "4th — Out"]

for i, gid in enumerate(group_ids):
    row = i // N_COLS + 1
    col = i % N_COLS + 1
    teams = group_standings[gid]

    team_names = [t["team"] for t in teams]
    pts_vals   = [t["pts"]  for t in teams]
    gd_vals    = [f"{t['gd']:+d}" for t in teams]
    colors     = QUAL_COLORS[:len(teams)]

    # Horizontal bar per team
    for j, (t, pts, gd, color) in enumerate(zip(team_names, pts_vals, gd_vals, colors)):
        adv = probs.get(t, {}).get("group_advance", 0)
        label = f"<b>{t}</b>  {pts}pts  GD{gd}  [{adv:.0f}% advance]"
        fig.add_trace(
            go.Bar(
                x=[pts], y=[t],
                orientation="h",
                marker_color=color,
                text=label,
                textposition="inside",
                insidetextanchor="start",
                hovertemplate=f"{t}: {pts} pts, GD {gd}<extra></extra>",
                showlegend=(i == 0 and j < 4),
                legendgroup=QUAL_LABELS[j] if j < 4 else "",
                name=QUAL_LABELS[j] if j < 4 else "",
            ),
            row=row, col=col,
        )

    fig.update_yaxes(autorange="reversed", row=row, col=col, showticklabels=False)
    fig.update_xaxes(range=[0, 10], row=row, col=col, showticklabels=False, showgrid=False)

fig.update_layout(
    title="<b>WC 2026 — Group Stage (Most Likely Standings)</b>",
    height=700,
    barmode="overlay",
    paper_bgcolor="#0e0e0e",
    plot_bgcolor="#1a1a1a",
    font_color="white",
    legend=dict(orientation="h", y=-0.04, x=0.5, xanchor="center"),
)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Tournament Bracket

Full bracket from Round of 32 through to the Final. Left side = Paths 1 & 2, right side = Paths 3 & 4.
Highlighted boxes are the predicted winners at each stage.

In [ ]:
def _bracket_fig(bracket: dict, probs: dict) -> go.Figure:
    """Draw the full 48-team knockout bracket as an annotated tree."""

    fig = go.Figure()
    annotations = []
    shapes = []

    # -----------------------------------------------------------------------
    # Layout constants
    # -----------------------------------------------------------------------
    # X positions per round (left side goes right→, right side mirror)
    # Rounds: R32=0, R16=1, QF=2, SF=3, Final=4
    X_LEFT  = [0,   1.5, 3.0, 4.5, 6.0]   # path 1&2 flow left→right
    X_RIGHT = [12, 10.5, 9.0, 7.5, 6.0]   # path 3&4 flow right→left
    BOX_W, BOX_H = 1.3, 0.35
    CANVAS_H = 36

    GOLD   = "#FFD700"
    SILVER = "#C0C0C0"
    GREEN  = "#27AE60"
    BLUE   = "#2980B9"
    GREY   = "#555555"
    BG     = "#1a1a1a"
    TEXT   = "#EEEEEE"

    champion = bracket.get("champion", "")
    finalist = bracket.get("finalist", "")
    third    = bracket.get("third", "")

    def _box_color(team: str, is_winner: bool) -> str:
        if team == champion:  return GOLD
        if team == finalist:  return SILVER
        if team == third:     return "#CD7F32"
        if is_winner:         return GREEN
        return GREY

    def _text_color(team: str, is_winner: bool) -> str:
        return "#000000" if team in (champion, finalist, third) or is_winner else TEXT

    def add_box(x: float, y: float, team: str, is_winner: bool, detail: str = "") -> None:
        fill = _box_color(team, is_winner)
        tc   = _text_color(team, is_winner)
        prob = probs.get(team, {}).get("champion", 0)
        label = team if len(team) <= 14 else team[:12] + "…"
        text  = f"<b>{label}</b>" + (f"<br><span style='font-size:9px'>{prob:.1f}% win</span>" if prob > 0 else "")
        shapes.append(dict(
            type="rect",
            x0=x - BOX_W/2, y0=y - BOX_H/2,
            x1=x + BOX_W/2, y1=y + BOX_H/2,
            fillcolor=fill, line=dict(color="#333", width=1),
        ))
        annotations.append(dict(
            x=x, y=y, text=text,
            showarrow=False,
            font=dict(size=9, color=tc),
            align="center",
        ))

    def add_line(x0, y0, x1, y1):
        shapes.append(dict(
            type="line",
            x0=x0, y0=y0, x1=x1, y1=y1,
            line=dict(color="#444", width=1),
        ))

    # -----------------------------------------------------------------------
    # Lay out left side (Paths 1 & 2) and right side (Paths 3 & 4)
    # -----------------------------------------------------------------------
    paths = bracket.get("paths", [])

    # Y positions: 4 paths × 8 teams = 32 R32 slots spread over canvas
    # Each path occupies canvas_H/4 vertical space
    PATH_H = CANVAS_H / 4

    r32_ys = {}  # match_key → (team, y)
    r32_winners: list[tuple[str, float]] = []  # (team, mid_y)

    for pi, path in enumerate(paths):
        is_right = pi >= 2
        x_cols = X_RIGHT if is_right else X_LEFT

        path_top = CANVAS_H - pi * PATH_H
        path_bot = path_top - PATH_H
        path_mid = (path_top + path_bot) / 2

        # 4 R32 matches → 8 teams
        r32 = path.get("r32", [])
        # Distribute 4 matches vertically within path_H
        slot_h = PATH_H / 4
        r32_slot_ys = [path_top - slot_h * (j + 0.5) for j in range(4)]

        r32_winner_ys = []
        for j, match in enumerate(r32):
            home, away, winner = match["home"], match["away"], match["winner"]
            loser = away if winner == home else home
            base_y = r32_slot_ys[j]
            y_home = base_y + BOX_H * 0.7
            y_away = base_y - BOX_H * 0.7

            add_box(x_cols[0], y_home, home, home == winner)
            add_box(x_cols[0], y_away, away, away == winner)
            add_line(x_cols[0] + BOX_W/2 * (1 if not is_right else -1), base_y,
                     x_cols[1] + BOX_W/2 * (-1 if not is_right else 1), base_y)
            r32_winner_ys.append((winner, base_y))

        # R16: 2 matches (pairs of r32 winners)
        r16 = path.get("r16", [])
        r16_winner_ys = []
        for j, match in enumerate(r16):
            home, away, winner = match["home"], match["away"], match["winner"]
            mid_y = (r32_winner_ys[j*2][1] + r32_winner_ys[j*2+1][1]) / 2
            add_box(x_cols[1], mid_y, winner, True)
            add_line(x_cols[1] + BOX_W/2 * (1 if not is_right else -1), mid_y,
                     x_cols[2] + BOX_W/2 * (-1 if not is_right else 1), mid_y)
            r16_winner_ys.append((winner, mid_y))

        # QF
        qf = path.get("qf", {})
        qf_winner = qf.get("winner", "")
        qf_y = (r16_winner_ys[0][1] + r16_winner_ys[1][1]) / 2
        add_box(x_cols[2], qf_y, qf_winner, True)
        add_line(x_cols[2] + BOX_W/2 * (1 if not is_right else -1), qf_y,
                 x_cols[3] + BOX_W/2 * (-1 if not is_right else 1), qf_y)

        r32_winners.append((qf_winner, qf_y))

    # -----------------------------------------------------------------------
    # Semi-finals & Final
    # -----------------------------------------------------------------------
    sf1 = bracket.get("sf1", {})
    sf2 = bracket.get("sf2", {})
    fin = bracket.get("final", {})

    sf1_y = (r32_winners[0][1] + r32_winners[1][1]) / 2
    sf2_y = (r32_winners[2][1] + r32_winners[3][1]) / 2

    add_box(X_LEFT[3], sf1_y, sf1.get("winner", ""), True)
    add_box(X_RIGHT[3], sf2_y, sf2.get("winner", ""), True)

    # Connect SF to Final
    final_y = (sf1_y + sf2_y) / 2
    add_line(X_LEFT[3]  + BOX_W/2, sf1_y, X_LEFT[4]  - BOX_W/2, final_y)
    add_line(X_RIGHT[3] - BOX_W/2, sf2_y, X_RIGHT[4] + BOX_W/2, final_y)

    # Final box — two halves
    add_box(X_LEFT[4] - 0.1, final_y + BOX_H * 0.7, fin.get("home", ""),
            fin.get("home") == champion)
    add_box(X_LEFT[4] - 0.1, final_y - BOX_H * 0.7, fin.get("away", ""),
            fin.get("away") == champion)

    # Champion crown
    annotations.append(dict(
        x=X_LEFT[4] - 0.1, y=final_y,
        text=f"<b>🏆 {champion}</b>",
        showarrow=False,
        font=dict(size=11, color=GOLD),
        bgcolor="rgba(0,0,0,0.6)",
        bordercolor=GOLD,
        borderwidth=1,
        align="center",
        yanchor="middle",
    ))

    # 3rd place box
    tp = bracket.get("third_place", {})
    tp_y = min(sf1_y, sf2_y) - 3
    add_box(X_LEFT[4] - 0.1, tp_y, tp.get("winner", ""), True)
    annotations.append(dict(
        x=X_LEFT[4] - 0.1, y=tp_y - BOX_H * 1.2,
        text="3rd place",
        showarrow=False,
        font=dict(size=8, color="#888"),
    ))

    # -----------------------------------------------------------------------
    # Path labels on the sides
    # -----------------------------------------------------------------------
    path_names = ["Path 1\n(A·B·C)", "Path 2\n(D·E·F)", "Path 3\n(G·H·I)", "Path 4\n(J·K·L)"]
    for pi, pname in enumerate(path_names):
        path_top = CANVAS_H - pi * PATH_H
        path_bot = path_top - PATH_H
        py = (path_top + path_bot) / 2
        px = X_LEFT[0] - 1.0 if pi < 2 else X_RIGHT[0] + 1.0
        annotations.append(dict(
            x=px, y=py, text=pname.replace("\n", "<br>"),
            showarrow=False,
            font=dict(size=8, color="#aaa"),
            align="center",
        ))

    # Round labels
    round_labels = ["R32", "R16", "QF", "SF", "Final"]
    for xi, label in enumerate(round_labels):
        for x in ([X_LEFT[xi]] if xi < 4 else []) + ([X_RIGHT[xi]] if xi < 4 else [X_LEFT[4] - 0.1]):
            annotations.append(dict(
                x=x, y=CANVAS_H + 0.5, text=f"<b>{label}</b>",
                showarrow=False,
                font=dict(size=9, color="#bbb"),
                align="center",
            ))

    fig.update_layout(
        title="<b>WC 2026 — Full Tournament Bracket</b>",
        shapes=shapes,
        annotations=annotations,
        xaxis=dict(range=[-1.5, 13.5], showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(range=[-2, CANVAS_H + 1.5], showgrid=False, zeroline=False, showticklabels=False),
        height=1100,
        paper_bgcolor="#0e0e0e",
        plot_bgcolor="#0e0e0e",
        font_color="white",
        margin=dict(l=60, r=60, t=60, b=20),
    )
    return fig

_bracket_fig(bracket, probs).show()

## Champion Probabilities

All 48 teams ranked by probability of winning the tournament.

In [ ]:
ranked = sorted(probs.items(), key=lambda x: x[1]["champion"], reverse=True)

teams_sorted  = [r[0] for r in ranked]
champ_probs   = [r[1]["champion"] for r in ranked]
finalist_prob = [r[1]["finalist"] for r in ranked]
top4_prob     = [r[1]["top4"]     for r in ranked]
top8_prob     = [r[1]["top8"]     for r in ranked]
adv_prob      = [r[1]["group_advance"] for r in ranked]

# Determine bar colour: gold=champion, silver=finalist, bronze=3rd
bar_colors = []
champ = bracket["champion"]
fin   = bracket["finalist"]
thrd  = bracket["third"]
for t in teams_sorted:
    if t == champ: bar_colors.append("#FFD700")
    elif t == fin: bar_colors.append("#C0C0C0")
    elif t == thrd: bar_colors.append("#CD7F32")
    else: bar_colors.append("#2980B9")

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Champion",
    x=teams_sorted, y=champ_probs,
    marker_color=bar_colors,
    text=[f"{v:.1f}%" for v in champ_probs],
    textposition="outside",
    hovertemplate="%{x}<br>Champion: %{y:.1f}%<extra></extra>",
))

fig.update_layout(
    title="<b>WC 2026 — Probability of Winning the Tournament</b>",
    xaxis_tickangle=-45,
    yaxis_title="Probability (%)",
    height=520,
    paper_bgcolor="#0e0e0e",
    plot_bgcolor="#1a1a1a",
    font_color="white",
    yaxis=dict(gridcolor="#333"),
    margin=dict(b=140),
)
fig.show()

## Round-by-Round Reach Probabilities (Top 16)

Stacked bar showing how far each of the top-16 favourites is expected to reach.

In [ ]:
top16 = ranked[:16]
teams16 = [r[0] for r in top16]

# Each band = incremental probability (group_advance → top8 → top4 → finalist → champion)
ga16   = np.array([r[1]["group_advance"] for r in top16])
top8_  = np.array([r[1]["top8"]          for r in top16])
top4_  = np.array([r[1]["top4"]          for r in top16])
fin_   = np.array([r[1]["finalist"]      for r in top16])
champ_ = np.array([r[1]["champion"]      for r in top16])

fig = go.Figure()

fig.add_trace(go.Bar(name="Group stage only",  x=teams16, y=ga16 - top8_,  marker_color="#555"))
fig.add_trace(go.Bar(name="Reaches QF",         x=teams16, y=top8_ - top4_, marker_color="#2980B9"))
fig.add_trace(go.Bar(name="Reaches SF",         x=teams16, y=top4_ - fin_,  marker_color="#27AE60"))
fig.add_trace(go.Bar(name="Reaches Final",      x=teams16, y=fin_ - champ_, marker_color="#C0C0C0"))
fig.add_trace(go.Bar(name="Wins tournament",    x=teams16, y=champ_,        marker_color="#FFD700"))

fig.update_layout(
    barmode="stack",
    title="<b>WC 2026 — Tournament Reach Probabilities (Top 16)</b>",
    yaxis_title="Probability (%)",
    height=500,
    paper_bgcolor="#0e0e0e",
    plot_bgcolor="#1a1a1a",
    font_color="white",
    yaxis=dict(gridcolor="#333"),
    legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),
)
fig.show()

## Podium Summary

In [ ]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

c = Console()
c.print(Panel(
    f"[bold gold1]🏆 Champion:  {bracket['champion']} ({probs[bracket['champion']]['champion']:.1f}% chance)[/bold gold1]\n"
    f"[bold]🥈 Finalist:  {bracket['finalist']} ({probs[bracket['finalist']]['champion']:.1f}% chance)[/bold]\n"
    f"[bold]🥉 3rd Place: {bracket['third']} ({probs[bracket['third']]['champion']:.1f}% chance)[/bold]",
    title="Most Likely Podium",
    border_style="gold1",
))

t = Table(title="Top 10 — Win Probabilities")
t.add_column("#",        style="dim", justify="right")
t.add_column("Team",     style="bold")
t.add_column("Champion", style="gold1",  justify="right")
t.add_column("Finalist", style="cyan",   justify="right")
t.add_column("Top 4",    style="green",  justify="right")
t.add_column("Top 8",    style="dim",    justify="right")

for rank, (team, p) in enumerate(ranked[:10], 1):
    t.add_row(
        str(rank), team,
        f"{p['champion']:.1f}%",
        f"{p['finalist']:.1f}%",
        f"{p['top4']:.1f}%",
        f"{p['top8']:.1f}%",
    )
c.print(t)